# Conclusion

## Findings

- Hourly demand was successfully created.
- Demand was calculated for each NYC pickup zone.
- The forecasting target variable is now available.
- The dataset is ready for feature engineering.

## Output Dataset

The final dataset contains:

- Hour
- Pickup Location ID
- Demand

## Next Step

Notebook 05: Feature Engineering

We will create time-based and lag-based features to improve forecasting performance.

---

# NYC Taxi Demand Forecasting

## Notebook 04: Demand Creation

### Objective

Create the target variable required for forecasting.

### Why?

Machine learning models cannot directly forecast millions of individual taxi trips.

Instead, we must transform trip-level data into a time-series dataset where:

- Each row represents a specific hour.
- Each row belongs to a specific NYC pickup zone.
- Demand represents the total number of taxi pickups.

### Expected Outcome

A structured dataset containing:

- Hour
- Pickup Zone
- Demand

This dataset will be used for feature engineering and forecasting model development.

---

# Import Libraries

## Purpose

Import the required libraries for data loading and processing.

## Description

These libraries will be used throughout the notebook for demand creation.

## Expected Outcome

Libraries imported successfully.

In [2]:
import pandas as pd
import numpy as np

# For Seeing All column not ...
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


# Step 2:Load Clean Dataset

## Purpose

Load the cleaned dataset prepared in the previous notebook.

## Description

The cleaned dataset contains validated trip records and will be used to generate hourly taxi demand.


In [3]:
df = pd.read_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/data/yellow_taxi_clean.parquet"
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


# Step 3 :Dataset Shape

## Purpose

Understand the size of the cleaned dataset.

## Description

This helps verify that the dataset was loaded correctly before demand creation.

## Expected Outcome

Total rows and columns displayed.

In [4]:
rows, cols = df.shape

print(f"Total Rows    : {rows:,}")
print(f"Total Columns : {cols}")

Total Rows    : 3,724,888
Total Columns : 20


In [5]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1643304,1,2026-01-20 09:26:52,2026-01-20 09:35:35,1.0,1.30,1.0,N,113,231,1,10.00,3.25,0.5,2.95,0.00,1.0,17.70,2.5,0.00,0.75
2466721,2,2026-01-30 09:26:54,2026-01-30 09:45:40,1.0,2.72,1.0,N,68,144,1,18.40,0.00,0.5,4.63,0.00,1.0,27.78,2.5,0.00,0.75
1266344,2,2026-01-15 18:15:20,2026-01-15 18:21:33,1.0,0.56,1.0,N,161,161,1,7.20,2.50,0.5,2.89,0.00,1.0,17.34,2.5,0.00,0.75
1690239,2,2026-01-20 17:04:03,2026-01-20 17:17:01,1.0,1.60,1.0,N,140,236,2,12.80,2.50,0.5,0.00,0.00,1.0,19.30,2.5,0.00,0.00
653017,2,2026-01-09 10:35:22,2026-01-09 10:47:23,1.0,2.14,1.0,N,238,262,1,13.50,0.00,0.5,3.50,0.00,1.0,21.00,2.5,0.00,0.00
361196,1,2026-01-05 21:52:57,2026-01-05 22:05:22,1.0,7.30,1.0,N,70,262,1,28.20,10.25,0.5,5.00,7.46,1.0,52.41,2.5,1.75,0.00
1334772,2,2026-01-16 12:03:43,2026-01-16 12:22:20,2.0,4.12,1.0,N,148,239,1,22.60,0.00,0.5,3.00,0.00,1.0,30.35,2.5,0.00,0.75
3084558,2,2026-01-16 02:02:15,2026-01-16 02:21:57,NaN,10.06,NaN,None,50,18,0,28.64,0.00,0.5,0.00,0.00,1.0,33.39,NaN,NaN,0.75
2224886,2,2026-01-27 16:02:45,2026-01-27 16:22:44,1.0,1.58,1.0,N,231,249,1,17.00,2.50,0.5,7.28,0.00,1.0,31.53,2.5,0.00,0.75
2745361,2,2026-01-04 07:17:03,2026-01-04 07:31:49,NaN,2.89,NaN,None,225,256,0,17.32,0.00,0.5,0.00,0.00,1.0,18.82,NaN,NaN,0.00


# Step 4:Create Hour Feature

## Purpose

Convert pickup timestamps into hourly intervals.

## Description

Demand forecasting will be performed on an hourly basis.

Therefore, pickup timestamps must be grouped into hourly buckets.


In [6]:
df['hour'] = (
    df['tpep_pickup_datetime']
    .dt.floor('h')
)

print("Hour column created successfully.")

Hour column created successfully.


# Step:5 Verify Required Columns

## Purpose

Ensure that the required columns exist before creating demand.

## Description

Demand creation requires:

- Hour
- Pickup Location ID

## Expected Outcome

Required columns successfully verified.

In [7]:
required_cols = [
    'hour',
    'PULocationID'
]

for col in required_cols:
    print(
        f"{col} Exists :",
        col in df.columns
    )

hour Exists : True
PULocationID Exists : True


# Step 6: Create Hourly Demand

## Purpose

Calculate the number of taxi pickups for each zone during each hour.

## Description

Demand is defined as the total number of trips originating from a pickup zone within a specific hour.

## Expected Outcome

A new dataset containing:

- Hour
- Pickup Zone
- Demand
---

In [8]:
hourly_demand = (
    df.groupby(['hour', 'PULocationID'])
    .size()
    .reset_index(
        name='demand'
    )
)

print("Hourly demand dataset created successfully.")

Hourly demand dataset created successfully.


# Step 7:Preview Demand Dataset

## Purpose

Inspect the newly created forecasting dataset.

## Description

Each row represents the demand for a specific pickup zone during a specific hour.



In [9]:
hourly_demand

,hour,PULocationID,demand
0,2025-12-31 23:00:00,113,1
1,2025-12-31 23:00:00,132,3
2,2025-12-31 23:00:00,163,1
3,2025-12-31 23:00:00,164,1
4,2026-01-01 00:00:00,3,2
...,...,...,...
123597,2026-01-31 23:00:00,261,29
123598,2026-01-31 23:00:00,262,59
123599,2026-01-31 23:00:00,263,175
123600,2026-01-31 23:00:00,264,5


# Step 8:Understand Demand

## Purpose

Verify how demand values are calculated.

## Description

Demand represents the count of taxi pickups within a zone during a given hour.



In [10]:
hourly_demand.sample(10)

,hour,PULocationID,demand
31058,2026-01-08 20:00:00,75,36
31828,2026-01-09 01:00:00,144,42
27848,2026-01-08 01:00:00,82,2
86841,2026-01-22 23:00:00,249,228
12785,2026-01-04 04:00:00,179,2
105381,2026-01-27 16:00:00,223,5
86342,2026-01-22 20:00:00,151,38
29408,2026-01-08 10:00:00,250,2
28252,2026-01-08 04:00:00,256,2
82800,2026-01-21 23:00:00,41,15


In [11]:
print("Descriptive Demand Statistics:")
hourly_demand['demand'].describe()

Descriptive Demand Statistics:


count    123602.000000
mean         30.136147
std          65.045224
min           1.000000
25%           2.000000
50%           5.000000
75%          20.000000
max         798.000000
Name: demand, dtype: float64

# Step 9:Unique Pickup Zones

## Purpose

Determine how many NYC pickup zones are represented in the forecasting dataset.

## Description

Each zone will have its own demand pattern.



In [12]:
zones = hourly_demand['PULocationID'].nunique()

print(f"Unique Pickup Zones : {zones}")

Unique Pickup Zones : 262


# Step 10:Time Range Validation

## Purpose

Verify the period covered by the forecasting dataset.

## Description

Understanding the available date range is important before feature engineering and model training.



In [13]:
print(
    "Start Date :",
    hourly_demand['hour'].min()
)

print(
    "End Date :",
    hourly_demand['hour'].max()
)

Start Date : 2025-12-31 23:00:00
End Date : 2026-02-01 00:00:00


# Save Demand Dataset

## Purpose

Save the forecasting dataset for future notebooks.

## Description

This dataset will be used during feature engineering and model development.



In [14]:
hourly_demand.to_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/data/hourly_demand.parquet",
    index=False
)

print("Hourly demand dataset saved successfully.")

Hourly demand dataset saved successfully.


In [16]:
hourly_demand['demand'].sum()

np.int64(3724888)